<a href="https://colab.research.google.com/github/BigFoots625/IntroductionMachineLearningwithPython/blob/main/Chapter6_AlgorithmChains_and_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Algorithm Chains and Pipelines**
In machine learning, we rarely apply a single algorithm to raw data. We typically build a sequence of processing steps, such as scaling, feature selection, and finally, model training.

Scikit-learn provides the `Pipeline` class to simplify the process of building these chains. More importantly, it ensures that our model evaluation (like cross-validation) remains statistically valid by preventing information from the test set from "leaking" into the training process.

In [1]:
!pip install mglearn
import mglearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.svm import SVC
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 10.6 MB/s eta 0:00:00


### **Theoretical Explanation: Data Leakage During Preprocessing**
Imagine you want to use `GridSearchCV` to find the best parameters for an SVM.
A very common, **fundamentally flawed** workflow looks like this:
1. Scale the entire dataset $X$ using `MinMaxScaler`.
2. Pass the scaled dataset into `cross_val_score` or `GridSearchCV`.

**Why is this wrong?** Cross-validation splits the data into training folds and validation folds. However, because you scaled the *entire* dataset in Step 1, the scaler used information (the minimum and maximum values) from the validation folds to scale the training folds.

Information from the validation data has "leaked" into the training process. Your cross-validation scores will be artificially high, giving you a false sense of security, and the model will perform worse on completely new, unseen production data.

To evaluate the model correctly, the splitting of the dataset during cross-validation must happen **before** any preprocessing. The `Pipeline` class solves this by wrapping the preprocessing and the model into a single object that cross-validation can split properly.

In [2]:
# Load dataset
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=0)

# --- THE WRONG WAY (Data Leakage) ---
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)

# The cross_val_score will split X_train_scaled. But X_train_scaled was built
# using information from the entire X_train dataset, including the validation folds!
svm = SVC()
scores = cross_val_score(svm, X_train_scaled, y_train, cv=5)
print(f"Flawed CV Score (Data Leakage): {np.mean(scores):.3f}")

# --- THE RIGHT WAY (Using a Pipeline) ---
from sklearn.pipeline import Pipeline

# We build a pipeline object. It takes a list of tuples: (name, estimator)
pipe = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC())])

# Now, we pass the raw, unscaled X_train to cross_val_score.
# Inside the CV loop, the pipeline will correctly fit the scaler ONLY on the training folds,
# and then apply it to the validation fold. Zero leakage!
scores_pipe = cross_val_score(pipe, X_train, y_train, cv=5)
print(f"Correct CV Score (Pipeline): {np.mean(scores_pipe):.3f}")

Flawed CV Score (Data Leakage): 0.981
Correct CV Score (Pipeline): 0.981


### **Theoretical Explanation: Grid Searches with Pipelines**
The true power of pipelines shines when combined with `GridSearchCV`. We want to search for the best hyperparameter $C$ and $\gamma$ for our SVM, but we must do it without leaking data.

Since our estimator is now a `Pipeline`, we need to tell `GridSearchCV` which parameter belongs to which step in the pipeline. We do this using a specific syntax: the name of the step, followed by two underscores `__`, followed by the parameter name.

For example, to search over the parameter `C` for the step named `svm`, the key in our parameter grid must be named `svm__C`.

In [3]:
# Define the parameter grid using the double-underscore syntax
param_grid = {'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
              'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

# Instantiate the grid search using the pipeline we built earlier
grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)

# Fit the grid search on the raw training data
grid.fit(X_train, y_train)

print(f"Best cross-validation accuracy: {grid.best_score_:.3f}")
print(f"Test set score: {grid.score(X_test, y_test):.3f}")
print(f"Best parameters: {grid.best_params_}")

Best cross-validation accuracy: 0.981
Test set score: 0.972
Best parameters: {'svm__C': 1, 'svm__gamma': 1}


### **Theoretical Explanation: The Pipeline Interface**
A `Pipeline` can have any number of steps. The only strict rule is that every step except the very last one must have a `transform` method (meaning it must be a preprocessing step like a scaler or PCA). The last step can be a classifier, regressor, or even another transformer.

**How `.fit()` works internally:**
When you call `pipe.fit(X)`, it calls `fit` on the first step, then `transform` on the first step, and passes that transformed data to the `fit` method of the second step, and so on, until it reaches the final step, where it just calls `fit`.

**How `.predict()` works internally:**
When you call `pipe.predict(X)`, it transforms the data through all the intermediate steps, and passes the final transformed data to the `predict` method of the final step.

**The `make_pipeline` Convenience Function:**
Creating pipelines with tuples (e.g., `[("scaler", MinMaxScaler()), ("svm", SVC())]`) can be tedious. Scikit-learn provides `make_pipeline`, which automatically names the steps for you based on the lowercase class name (e.g., `minmaxscaler`, `svc`).

In [4]:
from sklearn.pipeline import make_pipeline

# Create a pipeline without explicitly naming the steps
pipe_short = make_pipeline(MinMaxScaler(), SVC(C=100))
print(f"Automatically generated step names: {pipe_short.steps}\n")

# Fit the pipeline
pipe_short.fit(X_train, y_train)

# If we need to access a specific step inside the pipeline (e.g., to look at the SVM's support vectors),
# we can access it using the .named_steps attribute
svm_step = pipe_short.named_steps['svc']
print(f"Number of support vectors in the SVM step: {svm_step.n_support_}")

Automatically generated step names: [('minmaxscaler', MinMaxScaler()), ('svc', SVC(C=100))]

Number of support vectors in the SVM step: [26 30]


### **Theoretical Explanation: Grid-Searching Everything**
The Pipeline allows us to treat preprocessing steps as hyperparameters. We can ask `GridSearchCV` not just "what is the best $C$ for the SVM?", but also "should we use `StandardScaler` or `MinMaxScaler`?", or "how many polynomial features should we extract?".

Furthermore, we can even grid-search **which model to use entirely**. We can define a single pipeline and parameter grid that tests a Random Forest, an SVM, and a Ridge classifier, alongside all their respective parameters and preprocessing steps, in one giant, perfectly leak-proof search.

When we want to skip a step in the pipeline entirely (for example, tree-based models don't need scaling), we can assign that step the value `'passthrough'` in our parameter grid.

In [5]:
# Create a generic pipeline with placeholder steps
# We use StandardScaler and SVC as defaults, but we will overwrite them in the grid
pipe = Pipeline([('preprocessing', StandardScaler()), ('classifier', SVC())])

# We define a list of dictionaries for our parameter grid.
# Each dictionary tests an entirely different model and preprocessing strategy.
param_grid = [
    # Search 1: Test SVC with StandardScaler, varying C and gamma
    {'classifier': [SVC()],
     'preprocessing': [StandardScaler(), MinMaxScaler()],
     'classifier__C': [0.01, 0.1, 1, 10, 100],
     'classifier__gamma': [0.01, 0.1, 1, 10, 100]},

    # Search 2: Test RandomForest. Trees don't need scaling, so we set preprocessing to 'passthrough'
    {'classifier': [RandomForestClassifier(random_state=0)],
     'preprocessing': ['passthrough'],
     'classifier__n_estimators': [10, 100, 1000],
     'classifier__max_features': [1, 2, 3]}
]

# Run the massive grid search
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

print(f"Best params found across all models and preprocessing:\n{grid.best_params_}\n")
print(f"Best cross-validation score: {grid.best_score_:.3f}")
print(f"Test-set score: {grid.score(X_test, y_test):.3f}")

Best params found across all models and preprocessing:
{'classifier': SVC(), 'classifier__C': 10, 'classifier__gamma': 0.01, 'preprocessing': StandardScaler()}

Best cross-validation score: 0.986
Test-set score: 0.979


### **Chapter 6 Summary**
* **Objective:** We learned how to chain multiple processing steps together into a single cohesive estimator.
* **Key Concepts Covered:**
    * **Data Leakage:** The critical error of scaling or transforming data *before* cross-validation splits it, leading to falsely optimistic evaluations.
    * **The Pipeline Class:** Wrapping transformers and models together.
    * **Parameter Grid Syntax:** Using the `stepname__parameter` double-underscore notation to tune models inside a pipeline via `GridSearchCV`.
    * **Ultimate Flexibility:** Using a single pipeline to search over different preprocessing methods, hyperparameters, and entirely different machine learning algorithms simultaneously.
* **Takeaway:** From this point forward, you should almost never use `cross_val_score` or `GridSearchCV` without wrapping your preprocessing and model inside a `Pipeline`. It is the industry standard for robust, error-free machine learning engineering.